# 04 · Triple cruce: emocional × personajes × espacio

Combina las tres capas para responder: ¿qué carga afectiva trae cada personaje a cada tipo de espacio?

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import corpus, spatial_extraction, affect_loader, character_extraction
import cross_analysis, viz_heatmap, exporters

cfg = corpus.load_config('configs/sin_remedio.yaml')
parrafos = corpus.parse_from_config('data/corpus.txt', cfg)

with open(cfg['lugares_catalogo_path']) as f:
    cat_lug = json.load(f)
with open(cfg['personajes_catalogo_path']) as f:
    cat_per = json.load(f)

df_esp = spatial_extraction.extract(parrafos, cat_lug)
df_emo = affect_loader.load('data/extraccion_emocional.xlsx')
pers_idx = character_extraction.index_paragraphs(parrafos, cat_per)
cruce = cross_analysis.cross_spatial_affect(df_esp, df_emo)

polaridad_fn = cross_analysis.make_polarity_fn(cfg['polaridad'])
print('✓ Datos cargados')

## 1. Triple cruce

In [ ]:
df_triple = cross_analysis.triple_cross(
    cruce, pers_idx, polaridad_fn, cfg['categorias_sociales'])

# Filtrar al cruce válido (con personaje y categoría)
df_t = df_triple[(df_triple['Personaje']!='(sin personaje)') & (df_triple['Categoría']!='(sin clasificación)')]
print(f"Triple cruce: {len(df_t)} filas")
df_t.head(10)

## 2. Firma afectiva por personaje

In [ ]:
firma = df_t.pivot_table(
    index='Personaje', columns='Polaridad', values='Párrafo',
    aggfunc='nunique', fill_value=0
).reset_index()
firma['Total'] = firma.iloc[:,1:].sum(axis=1)
neg = firma.get('Negativa', 0); pos = firma.get('Positiva', 0)
firma['Idx_desencanto'] = ((neg - pos) / (neg + pos).replace(0, float('nan'))).round(3)
firma.sort_values('Total', ascending=False)

## 3. Heatmap personaje × categoría espacial

In [ ]:
pers_info = {p['canonico']: p for p in cat_per}
viz_heatmap.heatmap_personaje_categoria(
    df_t, pers_info,
    'outputs/heatmap_personaje_categoria.png',
    min_apariciones=4,
)

## 4. Exportar Excel

In [ ]:
exporters.to_excel({
    'Triple cruce': {'df': df_t, 'title': 'Triple cruce completo'},
    'Firma afectiva': {'df': firma.sort_values('Total', ascending=False),
                       'title': 'Firma afectiva por personaje',
                       'conditional': ['Idx_desencanto']},
}, 'outputs/04_triple_cruce.xlsx')